# 1: Imports, Path Setup & Load Model

In [13]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image
import json
import pandas as pd

import io
from PIL import ImageFilter
from keras import layers

# Determine repo root directory (two levels up from notebooks/isabel/)
REPO_ROOT = Path.cwd().parents[1] if Path.cwd().name == "isabel" else Path.cwd()

# Path to saved Keras model inside the artifacts directory
MODEL_PATH = REPO_ROOT / "artifacts" / "gluten_guard_efficientnet_full.keras"

# Directory containing test images categorized in subfolders
DATA_DIR = Path(
    "/home/isabelksommerfeld/code/lrnzgll/gluten-guard/data/"
    "food-101-predict-images-40-classes/food-101-predict-images-40-classes/"
)

# Validate that paths exist
assert MODEL_PATH.exists(), f"Model file not found at: {MODEL_PATH}"
assert DATA_DIR.exists(), f"Data directory not found at: {DATA_DIR}"

# Load trained EfficientNet model
print(f"Loading model from: {MODEL_PATH}")
model = tf.keras.models.load_model(MODEL_PATH)
print("Model loaded successfully!")

Loading model from: /home/isabelksommerfeld/code/lrnzgll/gluten-guard/artifacts/gluten_guard_efficientnet_full.keras
Model loaded successfully!


# 2: Infer Class Names & Define Prediction Function

In [ ]:
# Try to load class names from artifacts JSON if available
CLASSES_PATH = REPO_ROOT / "artifacts" / "class_names.json"

if CLASSES_PATH.exists():
    data = json.loads(CLASSES_PATH.read_text())
    class_names = data["class_names"] if isinstance(data, dict) else data
else:
    # Full list of 101 Food-101 classes in standard alphabetical order
    class_names = [
        'apple_pie', 'baby_back_ribs', 'baklava', 'beef_carpaccio', 'beef_tartare',
        'beet_salad', 'beignets', 'bibimbap', 'bread_pudding', 'breakfast_burrito',
        'bruschetta', 'caesar_salad', 'cannoli', 'caprese_salad', 'carrot_cake',
        'ceviche', 'cheesecake', 'cheese_plate', 'chicken_curry', 'chicken_quesadilla',
        'chicken_wings', 'chocolate_cake', 'chocolate_mousse', 'churros', 'clam_chowder',
        'club_sandwich', 'crab_cakes', 'creme_brulee', 'croque_madame', 'cup_cakes',
        'deviled_eggs', 'donuts', 'dumplings', 'edamame', 'eggs_benedict',
        'escargots', 'falafel', 'filet_mignon', 'fish_and_chips', 'foie_gras',
        'french_fries', 'french_onion_soup', 'french_toast', 'fried_calamari', 'fried_rice',
        'frozen_yogurt', 'garlic_bread', 'gnocchi', 'greek_salad', 'grilled_cheese_sandwich',
        'grilled_salmon', 'guacamole', 'gyoza', 'hamburger', 'hot_and_sour_soup',
        'hot_dog', 'huevos_rancheros', 'hummus', 'ice_cream', 'lasagna',
        'lobster_bisque', 'lobster_roll_sandwich', 'macaroni_and_cheese', 'macarons', 'miso_soup',
        'mussels', 'nachos', 'omelette', 'onion_rings', 'oysters',
        'pad_thai', 'paella', 'pancakes', 'panna_cotta', 'peking_duck',
        'pho', 'pizza', 'pork_chop', 'poutine', 'prime_rib',
        'pulled_pork_sandwich', 'ramen', 'ravioli', 'red_velvet_cake', 'risotto',
        'samosa', 'sashimi', 'scallops', 'seaweed_salad', 'shrimp_and_grits',
        'spaghetti_bolognese', 'spaghetti_carbonara', 'spring_rolls', 'steak', 'strawberry_shortcake',
        'sushi', 'tacos', 'takoyaki', 'tiramisu', 'tuna_tartare', 'waffles'
    ]

print(f"Loaded {len(class_names)} class names.")


def predict_image_top_k(image_input, k: int = 3): ####### MODIFIED (image_path -> image_input)
    """
    Load and preprocess an image (from Path/str or PIL Image),
    then return top-k predictions across all 101 classes.
    Note: EfficientNet's preprocessing is built into the model, so input images
    remain in standard [0, 255] float32 scale.
    """
    # Check if input is already a PIL Image or a file path
    if isinstance(image_input, Image.Image): ####### MODIFIED
        img = image_input.convert("RGB").resize((224, 224)) ####### MODIFIED
    else:
        img = Image.open(image_input).convert("RGB").resize((224, 224)) ####### MODIFIED

    img_array = np.array(img, dtype=np.float32)
    img_batch = np.expand_dims(img_array, axis=0)  # Shape: (1, 224, 224, 3)

    probabilities = model.predict(img_batch, verbose=0)[0]

    # Sort probabilities to get top-k predicted indices
    top_indices = np.argsort(probabilities)[::-1][:k]

    results = [(class_names[idx], probabilities[idx]) for idx in top_indices]
    return img, results

Loaded 101 class names.


# 3: Preprocessing Experiments -- Ageing & Data Augmentation (test-time, independent toggles)
Two separate hypotheses to test before running predictions:
   
   A) Ageing: make modern test photos resemble older Food-101 training images
      (lower resolution, blur, noise, JPEG artifacts)
   
   B) Data Augmentation: apply the SAME augmentation layers used during training
      (RandomFlip/Rotation/Zoom/Contrast), but now at test time
 
 Toggle each independently below to see which one (if any) helps accuracy.

### A) Ageing

In [15]:
def age_image(
    img: Image.Image,
    jpeg_quality: int = 40,
    downscale_factor: float = 0.5,
    blur_radius: float = 0.6,
    noise_sigma: float = 6.0,
    seed: int = None,
) -> Image.Image:
    """Simulate an older, lower-quality photo: downscale, blur, noise, JPEG recompress."""
    rng = np.random.default_rng(seed)
    w, h = img.size

    small = img.resize((max(1, int(w * downscale_factor)), max(1, int(h * downscale_factor))), Image.BILINEAR)
    img = small.resize((w, h), Image.BILINEAR)
    img = img.filter(ImageFilter.GaussianBlur(radius=blur_radius))

    arr = np.array(img, dtype=np.float32)
    arr = np.clip(arr + rng.normal(0, noise_sigma, arr.shape), 0, 255).astype(np.uint8)
    img = Image.fromarray(arr)

    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=jpeg_quality)
    buf.seek(0)
    return Image.open(buf).convert("RGB")

### B) Data Augmentation (same layers as training)

In [16]:

test_time_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.1),
], name="test_time_augmentation")

def augment_image(img: Image.Image, seed: int = None) -> Image.Image:
    """Apply the training-time augmentation layers at test time (training=True forces them to run)."""
    arr = np.expand_dims(np.array(img.convert("RGB"), dtype=np.float32), axis=0)
    augmented = test_time_augmentation(arr, training=True)
    return Image.fromarray(np.array(augmented[0]).astype(np.uint8))

In [23]:
# --- Toggle which step(s) to test -- set both False for raw/original images ---
USE_AGEING = False
USE_AUGMENTATION = True
# True
# False

def preprocess_for_test(img: Image.Image, seed: int = None) -> Image.Image:
    """Applies the toggled step(s), in order: ageing first, then augmentation."""
    if USE_AGEING:
        img = age_image(img, seed=seed)
    if USE_AUGMENTATION:
        img = augment_image(img, seed=seed)
    return img

# 4: Full Dataset Evaluation Table (Per-Class & Overall Summary)

In [24]:
# Collect ALL test images across all subfolders
all_images = list(DATA_DIR.rglob("*.jpg")) + list(DATA_DIR.rglob("*.png"))
print(f"Evaluating all {len(all_images)} images across all class subfolders...")

results = []

for idx, img_path in enumerate(all_images): ####### MODIFIED
    # Direct folder name as ground truth class (matches model class format)
    true_class = img_path.parent.name

    # Load image and pass it through the test preprocessing pipeline
    raw_img = Image.open(img_path).convert("RGB") ####### MODIFIED
    processed_img = preprocess_for_test(raw_img, seed=idx) ####### MODIFIED

    # Run fast prediction on the preprocessed image
    _, top_preds = predict_image_top_k(processed_img, k=1) ####### MODIFIED
    top1_class, top1_conf = top_preds[0]

    is_correct = (top1_class == true_class)

    results.append({
        "file_name": img_path.name,
        "true_class": true_class,
        "predicted_class": top1_class, ####### MODIFIED
        "correct": is_correct
    })

# Convert to DataFrame
df_results = pd.DataFrame(results)

# Display detailed prediction table for all images
#display(df_results[["file_name", "true_class", "predicted_class", "correct"]]) ####### MODIFIED

# Group statistics by class and collect all predicted labels as a list
class_summary = (
    df_results.groupby("true_class")
    .agg(
        total_images=("correct", "count"),
        correct_predictions=("correct", "sum"),
        accuracy=("correct", "mean"),
        all_predictions=("predicted_class", list) ####### MODIFIED
    )
    .reset_index()
)

# Sort classes alphabetically
class_summary = class_summary.sort_values(by="true_class", ascending=True)

# Format accuracy as percentage
class_summary["accuracy_pct"] = class_summary["accuracy"].map("{:.1%}".format)

# Calculate overall statistics
total_all = class_summary["total_images"].sum()
correct_all = class_summary["correct_predictions"].sum()
overall_acc = correct_all / total_all if total_all > 0 else 0

# Append overall summary row at the bottom
total_row = pd.DataFrame([{
    "true_class": "--- TOTAL / OVERALL ---",
    "total_images": total_all,
    "correct_predictions": correct_all,
    "accuracy": overall_acc,
    "accuracy_pct": f"{overall_acc:.1%}",
    "all_predictions": "-" ####### MODIFIED
}])

final_summary = pd.concat([class_summary, total_row], ignore_index=True)

# Display final summary table including predictions
display(final_summary[["true_class", "total_images", "correct_predictions", "accuracy_pct", "all_predictions"]]) ####### MODIFIED

Evaluating all 150 images across all class subfolders...


,true_class,total_images,correct_predictions,accuracy_pct,all_predictions
0,apple_pie,6,2,33.3%,"[chicken_quesadilla, apple_pie, carrot_cake, d..."
1,baby_back_ribs,4,2,50.0%,"[baby_back_ribs, foie_gras, baby_back_ribs, st..."
2,baklava,5,2,40.0%,"[churros, baklava, baklava, garlic_bread, choc..."
3,beef_carpaccio,5,4,80.0%,"[beef_carpaccio, beef_tartare, beef_carpaccio,..."
4,beef_tartare,5,4,80.0%,"[beef_tartare, beef_tartare, pad_thai, beef_ta..."
5,bruschetta,3,3,100.0%,"[bruschetta, bruschetta, bruschetta]"
6,cannoli,3,2,66.7%,"[cannoli, cheese_plate, cannoli]"
7,carrot_cake,4,4,100.0%,"[carrot_cake, carrot_cake, carrot_cake, carrot..."
8,ceviche,3,2,66.7%,"[ceviche, bibimbap, ceviche]"
9,cheesecake,4,4,100.0%,"[cheesecake, cheesecake, cheesecake, cheesecake]"
